In [1]:
############### import libraries ###############

import itertools
import joblib
import json
import os
from pathlib import Path
import random
from IPython.display import Image

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.utils import plot_model
from keras.callbacks import EarlyStopping, Callback, ModelCheckpoint

2026-02-24 10:27:47.637544: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
############### Function definition ###############
def ams(s, b, b_r):
    radicand = 2.0 * ((s + b + b_r) * np.log(1.0 + s / (b + b_r)) - s)
    return np.sqrt(max(radicand, 0.0))

def best_ams_from_pred(pred, y_true, w_true, wFactor, b_r, n_thr):
    best     = -np.inf
    best_thr = None
    for thr in np.linspace(0.0, 1.0, n_thr):
        s = w_true[(pred > thr) & (y_true == 1)].sum()
        b = w_true[(pred > thr) & (y_true == 0)].sum()
        score = ams(s * wFactor, b * wFactor, b_r = b_r)
        if score > best:
            best     = score
            best_thr = float(thr)
    return float(best), float(best_thr)

In [11]:
############### TRAINING AND TEST DATASETS PATHS ###############

def download(url, output) :
    if not os.path.exists(output):
        print(f"{output} not found. Downloading...")
        urllib.request.urlretrieve(url, output)
    else:
        print(f"{output} already exists. Skip download.")

download("https://drive.google.com/uc?id=1JPOVfYXJNXeBgG_V0auiuK5W2CvP4_td", "training.csv")
download("https://drive.google.com/uc?id=11W8QiL98fEqV7Xfbw-xgXCC7G-SM1yMb", "test.csv")

training_path = 'training.csv'
test_path     = 'test.csv'


############### SAVE MODEL AND SUBMISSION FILE PATHS ###############

SAVE_MODEL_DIR    = Path('best_model')
model_name   = 'model.keras'
imputer_name = 'imputer.joblib'
scaler_name  = 'scaler.joblib'
meta_name    = 'meta.json'

# save plot figures
SAVE_PIC_DIR = Path('out')
model_diagram_name = 'model_diagram.png'
dnn_output_name = 'dnn_output.png'
history_name = 'history.png'

os.makedirs(SAVE_PIC_DIR, exist_ok = True)

SAVE_SUBMISSION_DIR  = Path('submission')
save_submission_name = 'submission.csv'

os.makedirs(SAVE_MODEL_DIR, exist_ok = True)

training.csv already exists. Skip download.
test.csv already exists. Skip download.


In [12]:
n_thr_test_eval = 20001 # test dataset の AMS 評価のための閾値の数 (n_thr_test_eval - 1) 個の閾値を 0.0 から 1.0 の範囲で等間隔に設定して AMS 評価を行う

In [14]:
############### EVALUATION OF TEST DATASET ###############

# Load the test dataset and meta information
df_test = pd.read_csv(test_path)
with open(SAVE_MODEL_DIR / f"{meta_name}", "r") as f:
    meta = json.load(f)

important_idx = meta["important_idx"]
d = meta["d"]
K = meta["K"]
seeds = meta["seeds"]

# check if the test dataset has the "Label" and "Weight" column
has_test_label = "Label" in df_test.columns
has_test_weight = "Weight" in df_test.columns

y_test, w_test = None, None

if has_test_label and has_test_weight:
    y_test = (df_test["Label"] == "s").astype(int).values
    w_test = df_test["Weight"].values

X_test = df_test.drop(columns=["EventId", "Weight", "Label"], errors = "ignore").values

X_test = np.where(X_test == -999.0, np.nan, X_test)
miss_flag = np.isnan(X_test[:, important_idx].astype(np.float32)).astype(np.float32)
X_test = np.concatenate([X_test, miss_flag], axis = 1)

# ===== test 評価（Label と Weight がある場合のみ） =====
if has_test_label and has_test_weight:
    ######### モデルの予測を平均して test_pred を作成 #########
    test_pred_each_list = [] # それぞれの test dataset の予測値の配列
    for i, seed in enumerate(seeds):
        for fold in range(K):
            # model, imputer, scaler の読み取り
            model   = keras.models.load_model(SAVE_MODEL_DIR / f"seed_{seed}/fold_{fold}/{model_name}") # 学習したデータの model の読み込み
            imputer = joblib.load(SAVE_MODEL_DIR / f"seed_{seed}/fold_{fold}/{imputer_name}")          # 学習したデータの imputer の読み込み
            scaler  = joblib.load(SAVE_MODEL_DIR / f"seed_{seed}/fold_{fold}/{scaler_name}")           # 学習したデータの scaler の読み込み

            # model の構造を表示し, /out に model_diagram.png として保存
            if (i == 0 and fold == 0):
                model.summary()
                plot_model(
                    model,
                    to_file = SAVE_PIC_DIR / f"{model_diagram_name}",
                    show_shapes = True,
                    show_layer_names = True
                )
                Image(filename = SAVE_PIC_DIR / f"{model_diagram_name}")
                print(f"{SAVE_PIC_DIR}/{model_diagram_name} にモデルの構造を保存しました.\n")

            X_test_imp = imputer.transform(X_test)
            X_test_scaled = scaler.transform(X_test_imp[:, :d])
            X_test_reconstructed = np.concatenate([X_test_scaled, X_test_imp[:, d:]], axis = 1)

            # それぞれの seed, fold における test dataset の予測値を test_pred_each として保存して test_pred_each_list に追加
            test_pred_each = model.predict(X_test_reconstructed, batch_size = 8192, verbose = 0).ravel() # それぞれの seed, fold における test dataset の予測値
            test_pred_each_list.append(test_pred_each)

            # それぞれの seed, fold における test dataset の AMS 評価
            best_ams_test_10_each, best_thr_test_10_each = best_ams_from_pred(test_pred_each, y_test, w_test, wFactor = 1.0, b_r = 10.0, n_thr = n_thr_test_eval)
            best_ams_test_30_each, best_thr_test_30_each = best_ams_from_pred(test_pred_each, y_test, w_test, wFactor = 1.0, b_r = 30.0, n_thr = n_thr_test_eval)

            print(f"########### Prediction for test dataset with seed {seed} fold {fold} ###########")
            print(f"AMS: {best_ams_test_10_each:.5f} at threshold: {best_thr_test_10_each:.5f}\n")
    
    ######### AMS evaluation on test dataset #########

    test_pred = np.mean(np.stack(test_pred_each_list, axis = 0), axis = 0)
    best_ams_test_10, best_thr_test_10 = best_ams_from_pred(test_pred, y_test, w_test, wFactor = 1.0, b_r = 10.0, n_thr = n_thr_test_eval)
    best_ams_test_30, best_thr_test_30 = best_ams_from_pred(test_pred, y_test, w_test, wFactor = 1.0, b_r = 30.0, n_thr = n_thr_test_eval)
    print(f"########### Final AMS Result ###########")
    print("for b_r=10.0:")
    print(f"Best AMS: {best_ams_test_10:.5f} at threshold: {best_thr_test_10:.5f}\n")
    print("for b_r=30.0:")
    print(f"Best AMS: {best_ams_test_30:.5f} at threshold: {best_thr_test_30:.5f}")
elif has_test_label and not has_test_weight:
    print("test.csv に Weight カラムが無いので、test AMS の評価はスキップします。")
elif not has_test_label and has_test_weight:
    print("test.csv に Label カラムが無いので、test AMS の評価はスキップします。")
else:
    print("test.csv に Label, Weight カラムが無いので、test AMS の評価はスキップします。")

test.csv に Label, Weight カラムが無いので、test AMS の評価はスキップします。
